# Model Evaluation for Capsule Vision Inspection

This notebook evaluates the trained YOLO model on the test set.

## Metrics:
- Precision
- Recall
- F1
- mAP@50
- mAP@50-95
- Per-class metrics
- Confusion matrix
- Error analysis

In [ ]:
# Install dependencies
!pip install ultralytics matplotlib seaborn

import os
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ultralytics import YOLO

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

In [ ]:
# Configuration
MODEL_PATH = "runs/train/capsule_yolov8n/weights/best.pt"
DATA_YAML = "/content/dataset/data.yaml"
IMGSZ = 640
CONF_THRESHOLD = 0.25

CLASS_NAMES = [
    "Good",
    "Crack",
    "Scratch",
    "Faulty Imprint",
    "Poke",
    "Squeeze",
    "Contamination",
]

print(f"Model: {MODEL_PATH}")
print(f"Data: {DATA_YAML}")
print(f"Confidence threshold: {CONF_THRESHOLD}")

In [ ]:
# Load model and run validation
model = YOLO(MODEL_PATH)

metrics = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=IMGSZ,
    conf=CONF_THRESHOLD,
    verbose=True,
)

print("\n=== Overall Metrics ===")
print(f"Precision: {metrics.box.p:.4f}")
print(f"Recall: {metrics.box.r:.4f}")
print(f"mAP@50: {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")

In [ ]:
# Per-class metrics table
print("=== Per-Class Metrics ===")
print(f"{'Class':<20} {'Precision':>10} {'Recall':>10} {'F1':>10} {'mAP50':>10} {'mAP50-95':>10}")
print("-" * 70)

per_class = {}
for i, name in enumerate(CLASS_NAMES):
    p = float(metrics.box.p[i]) if i < len(metrics.box.p) else 0.0
    r = float(metrics.box.r[i]) if i < len(metrics.box.r) else 0.0
    ap50 = float(metrics.box.ap50[i]) if i < len(metrics.box.ap50) else 0.0
    ap = float(metrics.box.ap[i]) if i < len(metrics.box.ap) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    per_class[name] = {"precision": p, "recall": r, "f1": f1, "map50": ap50, "map": ap}
    print(f"{name:<20} {p:>10.4f} {r:>10.4f} {f1:>10.4f} {ap50:>10.4f} {ap:>10.4f}")

In [ ]:
# Confusion matrix
from ultralytics.utils.plotting import plot_confusion_matrix

# The validation run saves confusion matrix to runs/val/confusion_matrix.png
cm_path = Path("runs/val/confusion_matrix.png")
if cm_path.exists():
    img = plt.imread(str(cm_path))
    plt.figure(figsize=(12, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Confusion Matrix")
    plt.show()
else:
    print("Confusion matrix not found. Check runs/val/ directory.")

In [ ]:
# Error analysis: run inference on test images and analyze failures
import cv2
from pathlib import Path

test_dir = Path("/content/dataset/images/test")
label_dir = Path("/content/dataset/labels/test")

def read_labels(lbl_path):
    """Read YOLO label file."""
    labels = []
    if not lbl_path.exists():
        return labels
    with open(lbl_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) == 5:
                labels.append(int(parts[0]))
    return labels

errors = []
for img_path in sorted(test_dir.glob("*.*")):
    lbl_path = label_dir / (img_path.stem + ".txt")
    true_labels = read_labels(lbl_path)

    # Run inference
    results = model.predict(str(img_path), conf=CONF_THRESHOLD, verbose=False)
    pred_labels = [int(d.cls) for d in results[0].boxes] if results[0].boxes is not None else []

    # Check for errors
    if not true_labels and pred_labels:
        errors.append({
            "image": img_path.name,
            "true": "None",
            "pred": [CLASS_NAMES[i] for i in pred_labels],
            "type": "False Positive",
        })
    elif true_labels and not pred_labels:
        errors.append({
            "image": img_path.name,
            "true": [CLASS_NAMES[i] for i in true_labels],
            "pred": "None",
            "type": "False Negative",
        })
    elif true_labels and pred_labels:
        true_set = set(true_labels)
        pred_set = set(pred_labels)
        if true_set != pred_set:
            errors.append({
                "image": img_path.name,
                "true": [CLASS_NAMES[i] for i in true_labels],
                "pred": [CLASS_NAMES[i] for i in pred_labels],
                "type": "Misclassification",
            })

print(f"Total errors found: {len(errors)}")
print()
for err in errors[:20]:
    print(f"{err['type']:<20} {err['image']:<30} True: {err['true']}  Pred: {err['pred']}")

In [ ]:
# Error type distribution
error_types = Counter(e["type"] for e in errors)
print("=== Error Type Distribution ===")
for etype, count in error_types.most_common():
    print(f"{etype:<20} {count}")

# Plot
if error_types:
    plt.figure(figsize=(8, 5))
    plt.bar(error_types.keys(), error_types.values(), color="coral")
    plt.title("Error Type Distribution")
    plt.xlabel("Error Type")
    plt.ylabel("Count")
    plt.show()

In [ ]:
# Benchmark table
print("=== Model Benchmark ===")
print(f"{'Model':<20} {'Precision':>10} {'Recall':>10} {'mAP50':>10} {'mAP50-95':>10}")
print("-" * 60)
print(f"{'YOLOv8n':<20} {metrics.box.p:>10.4f} {metrics.box.r:>10.4f} {metrics.box.map50:>10.4f} {metrics.box.map:>10.4f}")

# Inference speed
print("\n=== Inference Speed ===")
print(f"Speed (ms/img): {metrics.speed.get('inference', 0):.2f}")
print(f"FPS: {1000 / metrics.speed.get('inference', 1):.1f}")

In [ ]:
# Visualize sample predictions
import random

test_images = list(test_dir.glob("*.*"))
samples = random.sample(test_images, min(8, len(test_images)))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, img_path in enumerate(samples):
    results = model.predict(str(img_path), conf=CONF_THRESHOLD, verbose=False)
    result_img = results[0].plot()
    axes[i].imshow(result_img)
    axes[i].axis("off")
    axes[i].set_title(img_path.name)

for i in range(len(samples), len(axes)):
    axes[i].axis("off")

plt.tight_layout()
plt.show()